In [0]:
pip install azure-storage-file-datalake azure-identity dotenv tqdm

In [0]:
%restart_python

In [0]:
# https://learn.microsoft.com/en-us/azure/storage/blobs/data-lake-storage-directory-file-acl-python

In [0]:
import os
import csv
import tqdm
from dotenv import load_dotenv
import json
import pandas as pd

from azure.storage.filedatalake import (
    DataLakeServiceClient,
    DataLakeDirectoryClient,
    FileSystemClient
)
from azure.identity import DefaultAzureCredential

load_dotenv()


In [0]:
def get_service_client_account_key(account_name, account_key) -> DataLakeServiceClient:
    account_url = f"https://{account_name}.dfs.core.windows.net"
    service_client = DataLakeServiceClient(account_url, credential=account_key)

    return service_client

service_client = get_service_client_account_key(
    account_name=os.getenv("STORAGE_ACCOUNT_NAME"),
    account_key=os.getenv("STORAGE_ACCOUNT_KEY")
)

file_systems = service_client.list_file_systems()

for fs in file_systems:
    print(fs.name)

In [0]:
BASE_LANDING_VOLUME_PATH = os.getenv("BASE_LANDING_VOLUME_PATH")
spotify_operational_data_path = BASE_LANDING_VOLUME_PATH + "/spotify_operational_data"
spotify_operational_data_state_path = spotify_operational_data_path + "/_state" # for storing state (to load incrementally)
data_folders = ["artist", "album", "track", "playback"]
num_processed_files = 0

os.makedirs(spotify_operational_data_state_path, exist_ok=True)

# get the container "landing"
file_system_client = service_client.get_file_system_client(
    file_system="landing"
)


# list all files in the container
for folder in data_folders:
    os.makedirs(f"{spotify_operational_data_path}/{folder}", exist_ok=True) # create the folder to contain the data

    # create json file to save the current state for incremental loading
    if os.path.exists(f"{spotify_operational_data_state_path}/processed_files.json"):
        with open(f"{spotify_operational_data_state_path}/processed_files.json", "r", encoding="utf-8") as f:
            text = f.read()
            processed_files = set(json.loads(text))
    else:
        processed_files = set()



    csv_files = [ f.name for f in file_system_client.get_paths(path=f"{folder}")
        if f.name.endswith(".csv") and f.name not in processed_files
    ] # retrieve all new .csv files in the folder 

    # print(f"In folder '{folder}', the files are: {csv_files}")
    
    # process these csv files
    for file in tqdm.tqdm(csv_files, desc=f"Processing files in folder '{folder}'", total=len(csv_files)):
        file_client = file_system_client.get_file_client(file)

        data = file_client.download_file().readall()

        df = pd.read_csv(pd.io.common.BytesIO(data))

        # save in volumes
        output_file = f"{spotify_operational_data_path}/{file}"
        df.to_csv(output_file, encoding="utf-8", quoting= csv.QUOTE_ALL, index=False) 
  
  
        print(f"✅ Data successfully exported to: {output_file}")



    processed_files.update(csv_files)
    num_processed_files += len(csv_files)

    with open(f"{spotify_operational_data_state_path}/processed_files.json", "w", encoding="utf-8") as f:
        f.write(json.dumps(sorted(processed_files)))

print(f"✅ IN TOTAL: {num_processed_files} FILES HAVE BEEN PROCESSED")
